# Food Ordering Behaviour & Consumer Trends
**Data Analytics & Machine Learning project** — Python · Pandas · NumPy · SQL (MySQL) · Matplotlib · Seaborn · Scikit-learn

Sections: 1 Setup · 2 Load & inspect · 3 Cleaning · 4 EDA · 5 Statistical tests · 6 SQL analysis · 7 Machine learning · 8 Clustering · 9 Summary

Run from the `notebooks/` folder. Charts are saved to `../outputs/charts/`.

## 1. Setup

In [ ]:
import warnings, sqlite3
warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, r2_score, mean_absolute_error, silhouette_score)

DATA_PATH = Path('../data/food_ordering_behavior_dataset.csv')
CHART_DIR = Path('../outputs/charts'); CHART_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
sns.set_theme(style='whitegrid')
PALETTE = ['#2B5FA8', '#E8892B', '#7B3FA0', '#2E8B57', '#4FA3D1', '#E0B04B']

def save(name):
    plt.tight_layout(); plt.savefig(CHART_DIR / f'{name}.png', dpi=170); plt.show()

## 2. Load and inspect the data

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
df.head()

In [ ]:
df.info()
df.describe().T

## 3. Data cleaning and validation

In [ ]:
checks = {
    'Missing values': int(df.isna().sum().sum()),
    'Duplicate rows': int(df.duplicated().sum()),
    'Duplicate order_id': int(df['order_id'].duplicated().sum()),
    'Unique customers': int(df['user_id'].nunique()),
    'Age range': (int(df.age.min()), int(df.age.max())),
}
for k, v in checks.items(): print(f'{k:20s}: {v}')

# Outlier check using the IQR rule
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return int(((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).sum())
print({c: iqr_outliers(df[c]) for c in ['order_value', 'delivery_fee', 'time_taken_to_order', 'age']})

In [ ]:
# Feature engineering: age bands (the dataset has ages 18-44 only)
df['age_group'] = pd.cut(df.age, bins=[0, 24, 40, 100],
                         labels=['Gen-Z (<=24)', 'Millennial (25-40)', 'Adult (41+)'])
df['age_group'].value_counts()

## 4. Exploratory data analysis

In [ ]:
print(f"Total order value : {df.order_value.sum():,.0f}")
print(f"Average order value: {df.order_value.mean():.2f}")
print(f"Average rating     : {df.rating_given.mean():.2f}")
print(f"Repeat-order rate  : {(df.is_repeat_order == 'Yes').mean():.2%}")

In [ ]:
orders_city = df.city.value_counts()
fig, ax = plt.subplots(figsize=(6.4, 3.8))
sns.barplot(x=orders_city.index, y=orders_city.values, color=PALETTE[0], ax=ax)
ax.set_ylim(0, orders_city.max()*1.12)
for i, v in enumerate(orders_city.values): ax.text(i, v*1.015, f'{v:,}', ha='center', fontsize=9)
ax.set_title('Orders by City'); ax.set_xlabel(''); ax.set_ylabel('Orders')
save('city')

In [ ]:
orders_cuisine = df.cuisine.value_counts()
fig, ax = plt.subplots(figsize=(6.4, 3.8))
sns.barplot(y=orders_cuisine.index, x=orders_cuisine.values, color=PALETTE[1], ax=ax)
ax.set_xlim(0, orders_cuisine.max()*1.12)
for i, v in enumerate(orders_cuisine.values): ax.text(v*1.01, i, f'{v:,}', va='center', fontsize=9)
ax.set_title('Orders by Cuisine'); ax.set_ylabel(''); ax.set_xlabel('Orders')
save('cuisine')

In [ ]:
meal_rev = df.groupby('meal_type').order_value.sum()
fig, ax = plt.subplots(figsize=(6.4, 3.8))
sns.barplot(x=meal_rev.index, y=meal_rev.values/1e6, color=PALETTE[3], ax=ax)
ax.set_ylim(0, meal_rev.max()/1e6*1.12)
for i, v in enumerate(meal_rev.values): ax.text(i, v/1e6*1.015, f'{v/1e6:.2f}M', ha='center', fontsize=9)
ax.set_title('Revenue by Meal Type (millions)'); ax.set_xlabel(''); ax.set_ylabel('Order value (M)')
save('meal_rev')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
sns.histplot(df.age, bins=27, color=PALETTE[2], ax=axes[0]); axes[0].set_title('Age distribution')
sns.histplot(df.order_value, bins=40, color=PALETTE[0], ax=axes[1])
axes[1].axvline(df.order_value.mean(), color='k', ls='--'); axes[1].set_title('Order value distribution')
save('distributions')

In [ ]:
num = df[['age', 'order_value', 'delivery_fee', 'time_taken_to_order', 'rating_given']].copy()
num['repeat'] = (df.is_repeat_order == 'Yes').astype(int)
num['discount'] = (df.discount_applied == 'Yes').astype(int)
corr = num.corr()
fig, ax = plt.subplots(figsize=(6.4, 4.2))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', vmin=-1, vmax=1, cbar=False, ax=ax)
ax.set_title('Correlation matrix'); save('corr')
print('Largest off-diagonal |correlation|:', corr.where(~np.eye(len(corr), dtype=bool)).abs().max().max().round(4))

In [ ]:
ct = pd.crosstab(df.city, df.cuisine)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu', cbar=False, ax=ax)
ax.set_title('City x Cuisine order counts'); ax.set_xlabel(''); ax.set_ylabel(''); plt.xticks(rotation=25)
save('city_cuisine')

In [ ]:
age_rest = pd.crosstab(df.age_group, df.restaurant_type)
fig, ax = plt.subplots(figsize=(6.4, 3.8))
age_rest.plot(kind='bar', stacked=True, ax=ax, color=[PALETTE[3], PALETTE[1], PALETTE[2]])
ax.set_title('Orders by age group and restaurant type'); ax.set_xlabel(''); plt.xticks(rotation=0)
save('age_rest')
age_rest

## 5. Statistical tests (alpha = 0.05)

In [ ]:
rows = []
for a, b in [('city','cuisine'), ('discount_applied','is_repeat_order'), ('mood','is_repeat_order'),
             ('restaurant_type','rating_given'), ('rainy_weather','rating_given')]:
    chi2, p, _, _ = stats.chi2_contingency(pd.crosstab(df[a], df[b]))
    rows.append(('Chi-square', f'{a} vs {b}', round(p, 3)))
for g in ['city', 'restaurant_type']:
    F, p = stats.f_oneway(*[x.order_value.values for _, x in df.groupby(g)])
    rows.append(('ANOVA', f'order_value by {g}', round(p, 3)))
t, p = stats.ttest_ind(df[df.discount_applied=='Yes'].order_value, df[df.discount_applied=='No'].order_value)
rows.append(('t-test', 'order_value: discount vs none', round(p, 3)))
tests = pd.DataFrame(rows, columns=['Test', 'Variables', 'p-value'])
tests['Significant?'] = np.where(tests['p-value'] < 0.05, 'Yes', 'No')
tests

## 6. SQL analysis
The same queries are in `../sql/03_business_queries.sql` for **MySQL**. Here they run on an in-memory SQLite copy so the notebook works without a database server; the syntax is standard and MySQL 8 compatible.

In [ ]:
con = sqlite3.connect(':memory:')
df.drop(columns='age_group').to_sql('food_orders', con, index=False)

QUERIES = {
'Q1 Orders, revenue and AOV by city': '''
SELECT city, COUNT(*) AS orders, ROUND(SUM(order_value)) AS revenue,
       ROUND(AVG(order_value), 2) AS avg_order_value
FROM food_orders GROUP BY city ORDER BY orders DESC;''',
'Q2 Revenue and rating by meal type': '''
SELECT meal_type, COUNT(*) AS orders, SUM(order_value) AS revenue,
       ROUND(AVG(rating_given), 3) AS avg_rating
FROM food_orders GROUP BY meal_type ORDER BY revenue DESC;''',
'Q3 Repeat-order rate by discount': '''
SELECT discount_applied, COUNT(*) AS orders,
       ROUND(100.0 * SUM(is_repeat_order = 'Yes') / COUNT(*), 2) AS repeat_rate_pct
FROM food_orders GROUP BY discount_applied;''',
'Q4 Top cuisine in each city': '''
SELECT city, cuisine, orders FROM (
  SELECT city, cuisine, COUNT(*) AS orders,
         RANK() OVER (PARTITION BY city ORDER BY COUNT(*) DESC) AS rk
  FROM food_orders GROUP BY city, cuisine
) t WHERE rk = 1;''',
'Q5 Top 5 AOV by restaurant type and company': '''
SELECT restaurant_type, company, ROUND(AVG(order_value), 2) AS aov
FROM food_orders GROUP BY restaurant_type, company ORDER BY aov DESC LIMIT 5;''',
}
for name, q in QUERIES.items():
    print('\n' + name); display(pd.read_sql(q, con))

## 7. Machine learning: supervised models

In [ ]:
base = df.drop(columns=['order_id', 'user_id', 'age_group'])

def evaluate_classifiers(X, y, name):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    sc = StandardScaler().fit(X_tr); X_tr_s, X_te_s, X_s = sc.transform(X_tr), sc.transform(X_te), sc.transform(X)
    models = {'Logistic Regression': LogisticRegression(max_iter=1000),
              'Decision Tree': DecisionTreeClassifier(max_depth=6, random_state=RANDOM_STATE),
              'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE)}
    avg = 'binary' if y.nunique() == 2 else 'macro'
    rows = []
    for mname, m in models.items():
        m.fit(X_tr_s, y_tr); pred = m.predict(X_te_s)
        row = {'Task': name, 'Model': mname,
               'Accuracy': accuracy_score(y_te, pred),
               'Precision': precision_score(y_te, pred, average=avg, zero_division=0),
               'Recall': recall_score(y_te, pred, average=avg, zero_division=0),
               'F1': f1_score(y_te, pred, average=avg, zero_division=0),
               'ROC-AUC': roc_auc_score(y_te, m.predict_proba(X_te_s)[:, 1]) if y.nunique() == 2 else np.nan,
               'CV accuracy (5-fold)': cross_val_score(m, X_s, y, cv=5).mean()}
        rows.append(row)
    res = pd.DataFrame(rows); res['Baseline (majority class)'] = y_te.value_counts(normalize=True).max()
    return res.round(3)

# 7.1 Repeat order (binary)
y_repeat = (df.is_repeat_order == 'Yes').astype(int)
X_repeat = pd.get_dummies(base.drop(columns='is_repeat_order'), drop_first=True).astype(float)
res_repeat = evaluate_classifiers(X_repeat, y_repeat, 'Repeat order'); res_repeat

In [ ]:
# 7.2 Rating (5-class)
X_rating = pd.get_dummies(base.drop(columns=['rating_given', 'is_repeat_order']), drop_first=True).astype(float)
res_rating = evaluate_classifiers(X_rating, df.rating_given, 'Rating 1-5'); res_rating

In [ ]:
# 7.3 Order value (regression)
X_val = pd.get_dummies(base.drop(columns=['order_value', 'is_repeat_order']), drop_first=True).astype(float)
X_tr, X_te, y_tr, y_te = train_test_split(X_val, df.order_value, test_size=0.2, random_state=RANDOM_STATE)
sc = StandardScaler().fit(X_tr)
reg_rows = []
for mname, m in {'Linear Regression': LinearRegression(),
                 'Random Forest': RandomForestRegressor(n_estimators=150, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE)}.items():
    m.fit(sc.transform(X_tr), y_tr); pred = m.predict(sc.transform(X_te))
    reg_rows.append({'Model': mname, 'R2': r2_score(y_te, pred), 'MAE': mean_absolute_error(y_te, pred)})
reg = pd.DataFrame(reg_rows)
reg['Baseline MAE (predict mean)'] = mean_absolute_error(y_te, np.full(len(y_te), y_tr.mean()))
reg.round(3)

In [ ]:
# Feature importance (Random Forest, repeat order)
rf = RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=RANDOM_STATE).fit(X_repeat, y_repeat)
imp = pd.Series(rf.feature_importances_, X_repeat.columns).sort_values().tail(8)
fig, ax = plt.subplots(figsize=(6.4, 3.8)); imp.plot(kind='barh', color=PALETTE[0], ax=ax)
ax.set_title('Random Forest feature importance (repeat order)'); save('feature_importance')

**Reading the results:** compare each model with its baseline. If accuracy is about equal to the majority-class share, ROC-AUC is about 0.50 and R² is about 0, the model has learned nothing useful beyond chance. Low-signal data cannot be fixed by tuning.

## 8. Customer segmentation (K-Means)

In [ ]:
features = ['age', 'order_value', 'delivery_fee', 'time_taken_to_order', 'rating_given']
Z = StandardScaler().fit_transform(df[features])
sil = {k: silhouette_score(Z, KMeans(k, n_init=10, random_state=RANDOM_STATE).fit_predict(Z),
                           sample_size=10000, random_state=RANDOM_STATE) for k in range(2, 8)}
fig, ax = plt.subplots(figsize=(6.4, 3.8)); ax.plot(list(sil), list(sil.values()), marker='o', color=PALETTE[2])
ax.set_ylim(0, 0.3); ax.set_xlabel('k'); ax.set_ylabel('Silhouette'); ax.set_title('K-Means: silhouette score by k')
save('silhouette'); sil

In [ ]:
km = KMeans(4, n_init=10, random_state=RANDOM_STATE).fit(Z)
df['cluster'] = km.labels_ + 1
profile = df.groupby('cluster')[features].mean().round(1)
profile['orders'] = df.cluster.value_counts().sort_index()
profile

## 9. Summary
* Data is clean: no missing values, duplicates or outliers.
* Orders are spread almost evenly across cities, cuisines and meal types (differences of 2-3%).
* None of the 8 statistical tests is significant at 5%; correlations between numeric features are about 0.
* Classification and regression models perform at chance level; K-Means shows weak structure (silhouette about 0.15).
* **Conclusion:** the attributes carry no usable predictive signal for repeat orders, ratings or order value. Richer data (item prices, order history, timestamps) is needed before building prediction tools.